# <font color="#418FDE" size="6.5" uppercase>**Verluste und Metriken**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Berechnen Regressionsverluste aus tatsächlichen Werten und festen Vorhersagen. 
- Erstellen einfache Baselines ohne Training und vergleichen sie mit Beispielvorhersagen. 
- Berechnen Klassifikationsmetriken, Schwellenwerte und probabilistische Kennzahlen manuell. 


## **1. Regressionsverluste verstehen**

### **1.1. Werte und Vorhersagen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_01_01.jpg?v=1787636413" width="250">



>* Regression sagt numerische Zielwerte voraus
>* Verlust misst Abweichung von Vorhersage und Realität

>* Werte und Vorhersagen paarweise zuordnen
>* Fehler entstehen aus passenden Wertepaaren

>* Herkunft der Vorhersage ist zweitrangig
>* Nähe zur Realität macht Leistung messbar



In [ ]:
#@title Python-Code - Werte und Vorhersagen

# Wir vergleichen echte Werte mit festen Vorhersagen.
# Jeder Fehler entsteht aus einem passenden Wertepaar.
# Die Grafik zeigt Abweichungen und mittlere Verluste.

import numpy as np
import matplotlib.pyplot as plt

# Diese kleinen Daten stehen für fünf Wohnungen.
actual_prices = np.array([300, 420, 500, 610, 700], dtype=float)
predicted_prices = np.array([320, 390, 530, 590, 760], dtype=float)

# Beide Reihen müssen gleich lang sein.
if actual_prices.shape != predicted_prices.shape:
    raise ValueError("Tatsächliche Werte und Vorhersagen passen nicht zusammen.")

# Fehler sind Vorhersage minus tatsächlicher Wert.
errors = predicted_prices - actual_prices
absolute_errors = np.abs(errors)
squared_errors = errors ** 2

# Zwei einfache Verluste fassen alle Paarfehler zusammen.
mae = np.mean(absolute_errors)
mse = np.mean(squared_errors)

print("Fall | echt | vorhergesagt | Fehler")
for index in range(len(actual_prices)):
    print(
        f"{index + 1} | {actual_prices[index]:.0f} | "
        f"{predicted_prices[index]:.0f} | {errors[index]:+.0f} Tsd. Euro"
    )

print(f"MAE: {mae:.1f} Tsd. Euro")
print(f"MSE: {mse:.1f} quadrierte Tsd. Euro")

# Die Punkte zeigen, welche Werte paarweise verglichen werden.
fig, ax = plt.subplots(figsize=(7, 4))
case_numbers = np.arange(1, len(actual_prices) + 1)

ax.plot(case_numbers, actual_prices, marker="o", label="Tatsächlich")
ax.plot(case_numbers, predicted_prices, marker="o", label="Vorhersage")

ax.set_title("Tatsächliche Werte und feste Vorhersagen")
ax.set_xlabel("Fall")
ax.set_ylabel("Preis in Tsd. Euro")
ax.set_xticks(case_numbers)
ax.legend()

plt.show()



### **1.2. Fehlerarten erkennen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_01_02.jpg?v=1787636411" width="250">



>* Fehler entstehen bei abweichenden Vorhersagen
>* Fehlerrichtung zeigt Unter- oder Überschätzung

>* Fehlergröße bestimmt die praktische Bedeutung
>* Abweichungen werden zu Verlustmaßen zusammengefasst

>* Ausreißer zeigen schlecht abgedeckte Datenbereiche.
>* Wiederkehrende Fehlermuster weisen auf Verzerrungen hin.



In [ ]:
#@title Python-Code - Fehlerarten erkennen

# Dieses Beispiel zeigt Fehlerarten bei Regressionsvorhersagen.
# Wir vergleichen Richtung und Größe einzelner Fehler.
# Am Ende werden typische Fehlermuster sichtbar.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Beispieldaten halten die Rechnung gut nachvollziehbar.
actual_prices = np.array([300, 350, 400, 450, 500], dtype=float)
predicted_prices = np.array([280, 370, 390, 520, 470], dtype=float)

# Gleiche Länge verhindert falsche Paarungen von Werten.
if actual_prices.shape != predicted_prices.shape:
    raise ValueError("Tatsächliche Werte und Vorhersagen müssen gleich lang sein.")

# Der Fehler behält die Richtung der Abweichung.
errors = predicted_prices - actual_prices
absolute_errors = np.abs(errors)

# Vorzeichen helfen beim Erkennen von Unter- und Überschätzung.
error_types = []
for error in errors:
    if error < 0:
        error_types.append("zu niedrig")
    elif error > 0:
        error_types.append("zu hoch")
    else:
        error_types.append("exakt")

# Diese Kennzahlen fassen Richtung und Größe zusammen.
mean_error = np.mean(errors)
mean_absolute_error = np.mean(absolute_errors)

print("Fall | Ist | Vorhersage | Fehler | Art")
for index in range(len(actual_prices)):
    print(
        f"{index + 1} | {actual_prices[index]:.0f} | "
        f"{predicted_prices[index]:.0f} | {errors[index]:+.0f} | "
        f"{error_types[index]}"
    )

print(f"Mittlerer Fehler: {mean_error:+.1f} Tsd. Euro")
print(f"Mittlerer absoluter Fehler: {mean_absolute_error:.1f} Tsd. Euro")

# Die Nulllinie trennt Überschätzung und Unterschätzung.
fig, ax = plt.subplots(figsize=(7, 4))
colors = np.where(errors >= 0, "tab:orange", "tab:blue")
ax.bar(range(1, len(errors) + 1), errors, color=colors)

ax.axhline(0, color="black", linewidth=1)
ax.set_title("Fehlerarten bei festen Regressionsvorhersagen")
ax.set_xlabel("Beispielfall")
ax.set_ylabel("Fehler in Tsd. Euro")
plt.show()



### **1.3. Verluste berechnen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_01_03.jpg?v=1787636415" width="250">



>* Vorhersagefehler pro Beobachtung bestimmen
>* Fehler zu vergleichbarem Gesamtverlust zusammenfassen

>* MAE misst Fehlergrößen in Zieleinheiten
>* Große Fehler können stärker gewichtet werden

>* Fehler schrittweise vergleichen, umformen und zusammenfassen
>* Verlustwerte immer fachlich interpretieren



In [ ]:
#@title Python-Code - Verluste berechnen

# Wir berechnen Verluste für feste Regressionsvorhersagen.
# Absolute und quadratische Fehler zeigen unterschiedliche Schwerpunkte.
# Die Ausgabe vergleicht zwei Vorhersagesätze verständlich.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Diese Werte stehen für tatsächliche Lieferzeiten in Minuten.
actual_values = np.array([30, 45, 50, 60, 80], dtype=float)
prediction_a = np.array([32, 43, 55, 58, 78], dtype=float)
prediction_b = np.array([30, 45, 50, 60, 95], dtype=float)

# Gleiche Längen sind für paarweise Fehler notwendig.
if not (len(actual_values) == len(prediction_a) == len(prediction_b)):
    raise ValueError("Alle Reihen müssen gleich lang sein.")

# Fehler sind Vorhersage minus tatsächlicher Wert.
error_a = prediction_a - actual_values
error_b = prediction_b - actual_values

# Absolute Fehler ignorieren die Richtung der Abweichung.
absolute_a = np.abs(error_a)
absolute_b = np.abs(error_b)

# Quadratische Fehler bestrafen große Abweichungen stärker.
squared_a = error_a ** 2
squared_b = error_b ** 2

# Mittelwerte fassen die einzelnen Fehler zusammen.
mae_a = np.mean(absolute_a)
mae_b = np.mean(absolute_b)
mse_a = np.mean(squared_a)
mse_b = np.mean(squared_b)

# Eine kleine Tabelle zeigt die ersten Rechenschritte.
loss_table = pd.DataFrame(
    {
        "Ist": actual_values,
        "Vorhersage A": prediction_a,
        "Fehler A": error_a,
        "|Fehler A|": absolute_a,
        "Fehler A²": squared_a,
    }
)

print("Einzelwerte für Vorhersage A:")
print(loss_table.round(1).to_string(index=False))
print(f"MAE A: {mae_a:.1f} Minuten, MSE A: {mse_a:.1f}")
print(f"MAE B: {mae_b:.1f} Minuten, MSE B: {mse_b:.1f}")

# Das Balkendiagramm vergleicht beide Verlustmaße.
metric_names = ["MAE", "MSE"]
values_a = [mae_a, mse_a]
values_b = [mae_b, mse_b]

x_positions = np.arange(len(metric_names))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x_positions - bar_width / 2, values_a, bar_width, label="Vorhersage A")
ax.bar(x_positions + bar_width / 2, values_b, bar_width, label="Vorhersage B")

ax.set_title("Regressionsverluste aus festen Vorhersagen")
ax.set_xlabel("Verlustmaß")
ax.set_ylabel("Verlustwert")
ax.set_xticks(x_positions)

ax.set_xticklabels(metric_names)
ax.legend()
plt.tight_layout()
plt.show()



## **2. Einfache Baselines**

### **2.1. Mittelwert als Baseline**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_02_01.jpg?v=1787636401" width="250">



>* Sagt immer den Trainingsdurchschnitt voraus
>* Dient als einfache Vergleichsmarke für Modelle

>* Mittelwert passt zu quadratischen Fehlermaßen
>* Baseline zeigt Nutzen zusätzlicher Merkmale

>* Mittelwert-Baseline als Mindestvergleich nutzen
>* Ausreißer können den Mittelwert verzerren



In [ ]:
#@title Python-Code - Mittelwert als Baseline

# Diese Übung zeigt eine Mittelwert-Baseline.
# Wir vergleichen sie mit festen Beispielvorhersagen.
# Niedrigere Fehler zeigen die bessere Vorhersage.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Trainingswerte liefern den konstanten Durchschnitt.
train_costs = np.array([82, 95, 101, 110, 117, 125], dtype=float)
mean_baseline = train_costs.mean()

# Testwerte simulieren neue Haushalte mit bekannten Kosten.
test_costs = np.array([90, 105, 130, 115], dtype=float)
baseline_predictions = np.full(test_costs.shape, mean_baseline)

# Diese festen Vorhersagen stehen für ein einfaches Beispielmodell.
model_predictions = np.array([92, 108, 122, 118], dtype=float)

# Die mittlere quadratische Abweichung bestraft große Fehler stärker.
baseline_mse = np.mean((test_costs - baseline_predictions) ** 2)
model_mse = np.mean((test_costs - model_predictions) ** 2)

# Eine kurze Prüfung verhindert unpassende Array-Längen.
if test_costs.shape != model_predictions.shape:
    raise ValueError("Testwerte und Vorhersagen müssen gleich lang sein.")

print(f"Trainingsmittelwert: {mean_baseline:.2f} Euro")
print(f"MSE der Mittelwert-Baseline: {baseline_mse:.2f}")
print(f"MSE der Beispielvorhersagen: {model_mse:.2f}")
print("Besser ist die Variante mit dem kleineren MSE.")

# Das Diagramm zeigt konstante und individuelle Vorhersagen.
case_numbers = np.arange(1, len(test_costs) + 1)
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(case_numbers, test_costs, marker="o", label="Tatsächliche Kosten")
ax.plot(case_numbers, baseline_predictions, marker="o", label="Mittelwert-Baseline")
ax.plot(case_numbers, model_predictions, marker="o", label="Beispielvorhersagen")

ax.set_title("Mittelwert-Baseline im Vergleich")
ax.set_xlabel("Testfall")
ax.set_ylabel("Monatliche Stromkosten in Euro")
ax.set_xticks(case_numbers)

ax.legend()
plt.show()



### **2.2. Median als Baseline**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_02_02.jpg?v=1787636404" width="250">



>* Median-Baseline sagt immer denselben typischen Wert voraus
>* Robust gegen Ausreißer und schiefe Verteilungen

>* Median nur aus Trainingsdaten bestimmen
>* Robuste Baseline für faire Modellvergleiche

>* Median-Baseline zeigt echte Modellverbesserung
>* Stark bei Ausreißern und absoluten Fehlern



In [ ]:
#@title Python-Code - Median als Baseline

# Wir vergleichen eine Median-Baseline mit Beispielvorhersagen.
# Der Median nutzt nur die Trainingszielwerte.
# Die Ausgabe zeigt faire Testverluste.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Trainingsdaten enthalten einen deutlichen Ausreißer.
train_prices = np.array([620, 650, 670, 690, 710, 730, 760, 2500])
test_prices = np.array([640, 680, 700, 720, 740])

# Diese Prüfung verhindert leere oder unpassende Zielwerte.
if train_prices.size == 0 or test_prices.size == 0:
    raise ValueError("Trainings- und Testwerte dürfen nicht leer sein.")

# Die Baseline wird ausschließlich aus Trainingsdaten berechnet.
median_prediction = np.median(train_prices)
mean_prediction = np.mean(train_prices)

# Jede Testwohnung erhält dieselbe konstante Vorhersage.
median_predictions = np.full(test_prices.shape, median_prediction)
mean_predictions = np.full(test_prices.shape, mean_prediction)

# Eine einfache Beispielvorhersage nutzt leicht angepasste Werte.
example_predictions = np.array([650, 675, 705, 735, 755])

# MAE misst die durchschnittliche absolute Abweichung.
median_mae = np.mean(np.abs(test_prices - median_predictions))
mean_mae = np.mean(np.abs(test_prices - mean_predictions))
example_mae = np.mean(np.abs(test_prices - example_predictions))

print(f"Median aus Trainingsdaten: {median_prediction:.0f} Euro")
print(f"Mittelwert aus Trainingsdaten: {mean_prediction:.0f} Euro")
print(f"MAE Median-Baseline: {median_mae:.1f} Euro")
print(f"MAE Mittelwert-Baseline: {mean_mae:.1f} Euro")
print(f"MAE Beispielvorhersage: {example_mae:.1f} Euro")

# Das Diagramm zeigt echte Werte und konstante Baselines.
fig, ax = plt.subplots(figsize=(7, 4))
positions = np.arange(1, test_prices.size + 1)

ax.plot(positions, test_prices, marker="o", label="Tatsächliche Testwerte")
ax.plot(positions, median_predictions, marker="s", label="Median-Baseline")
ax.plot(positions, mean_predictions, marker="^", label="Mittelwert-Baseline")

ax.set_title("Median-Baseline auf neuen Beispielen")
ax.set_xlabel("Testbeobachtung")
ax.set_ylabel("Miete in Euro")
ax.legend()

plt.show()



### **2.3. Mehrheitsklasse als Baseline**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_02_03.jpg?v=1787636402" width="250">



>* Häufigste Klasse immer vorhersagen
>* Naiver Mindestmaßstab für echte Modelle

>* Unausgeglichene Klassen können Genauigkeit täuschen
>* Baseline zeigt problematische Metriken und Modelle

>* Mehrheitsklasse relativiert scheinbar gute Genauigkeit
>* Nützliche Modelle erkennen wichtige Minderheitsfälle



In [ ]:
#@title Python-Code - Mehrheitsklasse als Baseline

# Wir bauen eine einfache Mehrheitsklassen-Baseline.
# Die häufigste Klasse wird immer vorhergesagt.
# Der Vergleich zeigt Nutzen und Grenzen.

import numpy as np
import matplotlib.pyplot as plt

# Diese Zielwerte stehen für echte Klassenzugehörigkeiten.
true_labels = np.array([
    "bestanden", "bestanden", "bestanden", "bestanden",
    "bestanden", "bestanden", "bestanden", "nicht bestanden",
    "nicht bestanden", "bestanden"
])

# Diese Beispielvorhersagen stammen aus einem gedachten Modell.
model_predictions = np.array([
    "bestanden", "bestanden", "nicht bestanden", "bestanden",
    "bestanden", "bestanden", "bestanden", "nicht bestanden",
    "bestanden", "bestanden"
])

# Wir prüfen, ob beide Listen gleich lang sind.
if len(true_labels) != len(model_predictions):
    raise ValueError("Die Listen müssen gleich lang sein.")

# Die Mehrheitsklasse ist die häufigste Zielklasse.
classes, counts = np.unique(true_labels, return_counts=True)
majority_class = classes[np.argmax(counts)]

# Die Baseline sagt für jeden Fall dieselbe Klasse voraus.
baseline_predictions = np.full(len(true_labels), majority_class)

# Genauigkeit ist der Anteil korrekter Vorhersagen.
baseline_accuracy = np.mean(baseline_predictions == true_labels)
model_accuracy = np.mean(model_predictions == true_labels)

# Für die Minderheitsklasse betrachten wir zusätzlich die Trefferquote.
minority_class = classes[np.argmin(counts)]
minority_mask = true_labels == minority_class

# Diese Kennzahl zeigt, ob seltene Fälle erkannt werden.
baseline_minority_recall = np.mean(
    baseline_predictions[minority_mask] == minority_class
)
model_minority_recall = np.mean(model_predictions[minority_mask] == minority_class)

print(f"Mehrheitsklasse: {majority_class}")
print(f"Baseline-Genauigkeit: {baseline_accuracy:.0%}")
print(f"Modell-Genauigkeit: {model_accuracy:.0%}")
print(f"Baseline-Trefferquote für '{minority_class}': {baseline_minority_recall:.0%}")
print(f"Modell-Trefferquote für '{minority_class}': {model_minority_recall:.0%}")

# Das Balkendiagramm vergleicht Baseline und Modell.
labels = ["Baseline", "Beispielmodell"]
accuracies = [baseline_accuracy, model_accuracy]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, accuracies, color=["lightgray", "steelblue"])
ax.set_ylim(0, 1)

ax.set_ylabel("Genauigkeit")
ax.set_title("Mehrheitsklasse als Baseline")
ax.bar_label(ax.containers[0], labels=[f"{value:.0%}" for value in accuracies])
plt.show()



## **3. Klassifikationsmetriken manuell berechnen**

### **3.1. Labels und Scores**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_03_01.jpg?v=1787636406" width="250">



>* Labels zeigen echte oder entschiedene Klassen.
>* Scores bewerten Zugehörigkeit vor der Entscheidung.

>* Scores sind nicht automatisch kalibrierte Wahrscheinlichkeiten
>* Schwellenwerte machen daraus feste Klassenentscheidungen

>* Schwellenwert steuert positive Vorhersagen
>* Labels, Scores und Schwellenwert dokumentieren



In [ ]:
#@title Python-Code - Labels und Scores

# Dieses Beispiel trennt Labels und Scores.
# Ein Schwellenwert erzeugt vorhergesagte Klassen.
# Die Ausgabe zeigt manuelle Klassifikationsmetriken.

import numpy as np
import matplotlib.pyplot as plt

# Diese kleinen Daten bleiben vollständig überschaubar.
true_labels = np.array([1, 0, 1, 0, 1, 0, 0, 1])
scores = np.array([0.90, 0.70, 0.65, 0.55, 0.45, 0.40, 0.20, 0.10])

# Diese Prüfung verhindert unpassende Eingabelängen.
if true_labels.shape != scores.shape:
    raise ValueError("Labels und Scores müssen gleich lang sein.")

# Der Schwellenwert macht aus Scores harte Entscheidungen.
threshold = 0.50
predicted_labels = (scores >= threshold).astype(int)

# Diese Zählungen bilden die Konfusionsmatrix manuell.
tp = int(np.sum((true_labels == 1) & (predicted_labels == 1)))
fp = int(np.sum((true_labels == 0) & (predicted_labels == 1)))
tn = int(np.sum((true_labels == 0) & (predicted_labels == 0)))
fn = int(np.sum((true_labels == 1) & (predicted_labels == 0)))

# Diese Formeln vermeiden Division durch null.
accuracy = (tp + tn) / len(true_labels)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

print(f"Schwellenwert: {threshold:.2f}")
print(f"Vorhergesagte Labels: {predicted_labels.tolist()}")
print(f"TP={tp}, FP={fp}, TN={tn}, FN={fn}")
print(f"Genauigkeit={accuracy:.2f}, Präzision={precision:.2f}, Sensitivität={recall:.2f}")

# Die Grafik zeigt Scores, Labels und Schwellenwert zusammen.
fig, ax = plt.subplots(figsize=(7, 4))
colors = np.where(true_labels == 1, "tab:orange", "tab:blue")

ax.scatter(range(len(scores)), scores, c=colors, s=90, label="Score je Beobachtung")
ax.axhline(threshold, color="black", linestyle="--", label="Schwellenwert")
ax.set_title("Aus Scores werden vorhergesagte Labels")
ax.set_xlabel("Beobachtung")

ax.set_ylabel("Score für die positive Klasse")
ax.set_ylim(0, 1)
ax.legend()
plt.show()



### **3.2. Konfusionsmatrix verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_03_02.jpg?v=1787636408" width="250">



>* Vergleicht wahre und vorhergesagte Klassen
>* Zeigt Fehlerarten trotz gleicher Genauigkeit

>* Wahre und vorhergesagte Labels systematisch zählen
>* Positive Klasse vor Metriken eindeutig festlegen

>* Schwellenwerte verändern positive und negative Vorhersagen
>* Fehlerfolgen hängen stark vom Anwendungskontext ab



In [ ]:
#@title Python-Code - Konfusionsmatrix verstehen

# Wir zählen Klassifikationsergebnisse Schritt für Schritt.
# Die vier Felder bilden die Konfusionsmatrix.
# Eine Grafik zeigt die gezählten Fehlerarten.

import numpy as np
import matplotlib.pyplot as plt

# Diese kleinen Listen sind unsere vollständigen Beispieldaten.
true_labels = np.array([1, 0, 1, 1, 0, 0, 1, 0, 1, 0])
predicted_labels = np.array([1, 0, 0, 1, 1, 0, 1, 0, 0, 0])

# Diese Prüfung verhindert unpassende Listenlängen.
if true_labels.shape != predicted_labels.shape:
    raise ValueError("Wahre und vorhergesagte Labels müssen gleich lang sein.")

# Die positive Klasse ist hier bewusst als Eins festgelegt.
positive_class = 1
negative_class = 0

# Jede Bedingung zählt genau eine Ergebnisart.
tp = int(np.sum((true_labels == positive_class) & (predicted_labels == positive_class)))
tn = int(np.sum((true_labels == negative_class) & (predicted_labels == negative_class)))

# Falsch positive und falsch negative Fälle sind unterschiedliche Fehler.
fp = int(np.sum((true_labels == negative_class) & (predicted_labels == positive_class)))
fn = int(np.sum((true_labels == positive_class) & (predicted_labels == negative_class)))

# Die Matrix ordnet Zeilen nach wahren Klassen.
confusion_matrix = np.array([[tn, fp], [fn, tp]])

# Kurze Ausgaben verbinden Begriffe mit Zahlen.
print("Positive Klasse: 1")
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"Konfusionsmatrix: [[{tn}, {fp}], [{fn}, {tp}]]")

# Die Grafik macht die vier Felder sichtbar.
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(confusion_matrix, cmap="Blues")

# Achsenbeschriftungen zeigen die Leserichtung der Matrix.
ax.set_title("Konfusionsmatrix manuell gezählt")
ax.set_xlabel("Vorhergesagte Klasse")
ax.set_ylabel("Tatsächliche Klasse")

# Die Klassenlabels bleiben bewusst einfach.
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["0", "1"])
ax.set_yticklabels(["0", "1"])

# Jede Zelle erhält ihren Namen und Zählwert.
cell_labels = np.array([["TN", "FP"], ["FN", "TP"]])
for row in range(2):
    for col in range(2):
        text = f"{cell_labels[row, col]}\n{confusion_matrix[row, col]}"
        ax.text(col, row, text, ha="center", va="center", color="black")

# Eine Farbskala hilft beim Vergleichen der Häufigkeiten.
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()



### **3.3. Passende Metriken wählen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_03_03.jpg?v=1787636409" width="250">



>* Fehlerkosten bestimmen die passende Metrik
>* Genauigkeit kann bei Ungleichgewicht täuschen

>* Schwellenwerte machen Scores zu Klassenentscheidungen
>* Kurven zeigen Verhalten über viele Schwellenwerte

>* Wahrscheinlichkeiten müssen verlässlich kalibriert sein
>* Metrik nach fachlichem Ziel auswählen



In [ ]:
#@title Python-Code - Passende Metriken wählen

# Dieses Beispiel vergleicht Metriken bei zwei Schwellenwerten.
# Es zeigt Zielkonflikte zwischen Recall und Präzision.
# Am Ende wird eine passende Metrik begründet.

import numpy as np
import matplotlib.pyplot as plt

# Kleine feste Daten machen die Rechnung gut nachvollziehbar.
true_labels = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
predicted_scores = np.array([0.95, 0.80, 0.55, 0.35, 0.70, 0.60, 0.45, 0.30, 0.20, 0.10])

# Diese Prüfung verhindert unpassende Eingaben für die Metriken.
if true_labels.shape != predicted_scores.shape:
    raise ValueError("Labels und Scores müssen gleich lang sein.")

# Zwei Schwellenwerte zeigen unterschiedliche fachliche Entscheidungen.
thresholds = [0.50, 0.75]
metric_rows = []

# Jede harte Entscheidung entsteht aus Score und Schwellenwert.
for threshold in thresholds:
    predicted_labels = (predicted_scores >= threshold).astype(int)
    tp = int(np.sum((true_labels == 1) & (predicted_labels == 1)))
    fp = int(np.sum((true_labels == 0) & (predicted_labels == 1)))

    fn = int(np.sum((true_labels == 1) & (predicted_labels == 0)))
    tn = int(np.sum((true_labels == 0) & (predicted_labels == 0)))
    accuracy = (tp + tn) / len(true_labels)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    metric_rows.append([threshold, accuracy, precision, recall, false_positive_rate])

# Die Ausgabe bleibt kurz und fokussiert.
print("Schwelle | Genauigkeit | Präzision | Recall | Falsch-Positiv-Rate")
for row in metric_rows:
    print(f"{row[0]:.2f}     | {row[1]:.2f}        | {row[2]:.2f}      | {row[3]:.2f}   | {row[4]:.2f}")

# Die Empfehlung hängt vom fachlichen Fehlerkosten-Szenario ab.
print("Bei seltenen Krankheiten ist oft hoher Recall wichtiger.")
print("Bei teuren Fehlalarmen ist oft hohe Präzision wichtiger.")

# Die Grafik macht den Zielkonflikt sichtbar.
fig, ax = plt.subplots(figsize=(7, 4))
metric_names = ["Genauigkeit", "Präzision", "Recall", "FPR"]
x_positions = np.arange(len(metric_names))

# Balken vergleichen beide Schwellenwerte nebeneinander.
bar_width = 0.35
low_values = metric_rows[0][1:]
high_values = metric_rows[1][1:]

ax.bar(x_positions - bar_width / 2, low_values, bar_width, label="Schwelle 0.50")
ax.bar(x_positions + bar_width / 2, high_values, bar_width, label="Schwelle 0.75")
ax.set_xticks(x_positions, metric_names)

ax.set_ylim(0, 1.05)
ax.set_ylabel("Metrikwert")
ax.set_title("Metriken ändern sich mit dem Schwellenwert")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Verluste und Metriken**</font>


In this lecture, you learned to:
- Berechnen Regressionsverluste aus tatsächlichen Werten und festen Vorhersagen. 
- Erstellen einfache Baselines ohne Training und vergleichen sie mit Beispielvorhersagen. 
- Berechnen Klassifikationsmetriken, Schwellenwerte und probabilistische Kennzahlen manuell. 

In the next Module (Module 7), we will go over 'Lineare Regression'